# AI Weekly Events Planner (Week 1 exercise)

Scrape **two** public listing pages about events (defaults: **Eventbrite Science & Tech** discovery URLs), extract **structured JSON** events with an LLM—**technology-related items only**—then generate a **personalized 7-day plan**.

Override sources with a space-separated env var, for example  
`EVENT_SOURCE_URLS="https://www.eventbrite.com/d/united-states/new-york--new-york/science-and-tech--events/ https://..."`.

The bare Eventbrite homepage lists almost no events in HTML; use **`/d/.../science-and-tech--events/`** (or your city’s discovery path). Automated scraping may conflict with Eventbrite’s terms—use **low volume / learning** only; production apps should use the [Eventbrite API](https://www.eventbrite.com/platform/api).

**Backends:** **OpenAI** (cloud API key) or **Ollama** (local) — same SDK; Ollama uses an OpenAI-compatible `/v1` endpoint. See [`day1_with_ollama.ipynb`](../../../solutions/day1_with_ollama.ipynb).


In [ ]:
import json
import os
import re
import sys
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(override=True)


WEEK1_DIR = Path.cwd().parents[2]
sys.path.append(str(WEEK1_DIR))

_LISTING_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    ),
}


def fetch_listing_text(url: str, max_chars: int | None = None) -> str:
    """Plain-text snapshot of a listing page (aligned with week1/scraper.py, larger cap for dense feeds)."""
    if max_chars is None:
        max_chars = int(os.getenv("SCRAPE_MAX_CHARS", "20000"))
    resp = requests.get(url, headers=_LISTING_HEADERS, timeout=45)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.content, "html.parser")
    title_el = soup.title.string.strip() if soup.title and soup.title.string else "No title found"
    if soup.body:
        for tag in soup.body(["script", "style", "img", "input"]):
            tag.decompose()
        body_text = soup.body.get_text(separator="\n", strip=True)
    else:
        body_text = ""
    return (title_el + "\n\n" + body_text)[:max_chars]


DEFAULT_SOURCE_URLS = (
    "https://www.eventbrite.com/d/united-states/new-york--new-york/science-and-tech--events/",
)


def _ollama_api_base_url() -> str:
    raw = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434').rstrip('/')
    if raw.endswith('/v1'):
        return raw
    return raw + '/v1'


def get_openai_compatible_client_and_model():
    provider = os.getenv('LLM_PROVIDER', '').strip().lower()
    api_key_present = bool(os.getenv('OPENAI_API_KEY'))

    if provider == 'openai' and not api_key_present:
        raise ValueError(
            "LLM_PROVIDER=openai but OPENAI_API_KEY is missing. "
            "Set it or run with LLM_PROVIDER=ollama and a local model."
        )

    use_ollama = provider == 'ollama' or (provider != 'openai' and not api_key_present)

    if not use_ollama:
        client = OpenAI()
        model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
        return client, model, 'openai'

    client = OpenAI(
        base_url=_ollama_api_base_url(),
        api_key=os.getenv('OLLAMA_API_KEY', 'ollama'),
    )
    model = os.getenv('OLLAMA_MODEL', 'llama3.2')
    return client, model, 'ollama'


client, MODEL, BACKEND_LABEL = get_openai_compatible_client_and_model()
print(f"LLM backend: {BACKEND_LABEL} | model={MODEL}")


def llm_chat(system: str, user: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': user},
        ],
        temperature=0.2,
    )
    return (resp.choices[0].message.content or '').strip()


## Fetch pages → plain text


In [ ]:
_raw = os.getenv('EVENT_SOURCE_URLS', '').split()
SOURCE_URLS = [u.strip() for u in _raw if u.strip()] or list(DEFAULT_SOURCE_URLS)

scraped_blocks = []
for url in SOURCE_URLS:
    try:
        snippet = fetch_listing_text(url)
        scraped_blocks.append(f'### Source: {url}\n\n{snippet}')
        print(url, "| chars:", len(snippet))
    except Exception as e:
        print("FAILED", url, e)

bundle = '\n\n---\n\n'.join(scraped_blocks)
if not bundle.strip():
    raise RuntimeError("No pages fetched — check URLs and connectivity.")

display(Markdown(f"Combined crawl: **{len(bundle)}** chars"))
print(bundle[:2200], "\n...(truncated)")


## Structured extraction (JSON array)

Prompt enforces JSON only; fenced output is stripped before `json.loads`. Only **technology-related** listings are extracted (software, AI/ML, data, infra, product/engineering, IT, automation, tech startups).


In [ ]:
EVENT_SCHEMA_HINT = (
    'Keys per extracted event object:\n'
    '  title: string\n'
    '  start_datetime_iso: RFC3339 or empty\n'
    '  end_datetime_iso: string or empty\n'
    '  location: string or empty\n'
    '  is_free: boolean or null\n'
    '  price_notes: string\n'
    '  source_url: string\n'
    '  summary: short sentence\n'
    '  tags: JSON array of short strings'
)

EXTRACTOR_SYSTEM = (
    'Extract upcoming conference/event items from noisy webpage text.\n'
    'Return ONLY a JSON array of objects. No markdown fences, no prose.\n'
    + EVENT_SCHEMA_HINT
    + '\nScope — technology only:\n'
    + '- Include items clearly about computing, software, AI/ML, data, cloud/DevOps, '
    + 'cybersecurity, product/engineering, hardware/engineering tech, IT, automation, '
    + 'or tech-industry professional gatherings.\n'
    + '- Omit parties, generic nightlife, theater, sports, pure biology/nature talks, '
    + 'or listings with no plausible tech angle unless the title explicitly ties to tech.\n'
    + '\nRules:\n'
    + '- Ground every title in snippets from the bundle; skip nav/chrome.\n'
    + '- Avoid inventing URLs; reuse listed links when evident.\n'
    + '- Normalize dates only when anchored in supplied text; resolve relative phrases '
    + '("Tomorrow", "This weekend", weekday-only) using the user message "Today\'s UTC RFC3339" '
    + 'as the reference instant.\n'
)


def strip_json_fence(text: str) -> str:
    text = text.strip()
    if text.startswith('```'):
        parts = text.split('```')
        if len(parts) >= 2:
            inner = parts[1]
            if inner.lstrip().lower().startswith('json'):
                inner = inner.lstrip()[4:].lstrip()
            return inner.strip()
    return text


def parse_events_json(raw: str) -> list:
    raw = strip_json_fence(raw)
    data = json.loads(raw)
    if not isinstance(data, list):
        raise TypeError("Expected a JSON array")
    return data


def llm_extract_events(page_bundle: str, sources: list[str]) -> list[dict[str, Any]]:
    ts = datetime.now(timezone.utc).isoformat()
    user = (
        'Source URLs:\n' + json.dumps(sources, indent=2)
        + f"\n\nToday's UTC RFC3339: {ts}\n\n"
        + 'Webpage-derived text bundle:\n---\n'
        + page_bundle[:20000]
        + '\n---\n'
    )
    raw = llm_chat(EXTRACTOR_SYSTEM, user)
    try:
        events = parse_events_json(raw)
    except json.JSONDecodeError:
        repair = llm_chat("Return ONLY fixed valid JSON array, no prose.", "Broken:\n" + raw[:14000])
        events = parse_events_json(repair)
    return events


events = llm_extract_events(bundle, SOURCE_URLS)
print("Extracted", len(events), "raw events")
print(json.dumps(events[:min(3, len(events))], indent=2)[:2500])


## Normalize, validate, dedupe


In [ ]:
@dataclass
class Event:
    title: str
    start_datetime_iso: str
    end_datetime_iso: str
    location: str
    is_free: bool | None
    price_notes: str
    source_url: str
    summary: str
    tags: list[str]


def normalize_event(d: dict) -> Event | None:
    title = str(d.get('title') or '').strip()
    if not title:
        return None
    tags_raw = d.get('tags') or []
    if not isinstance(tags_raw, list):
        tags_raw = []
    tags = [str(t).strip() for t in tags_raw if str(t).strip()]
    fv = d.get('is_free')
    if fv is not None and not isinstance(fv, bool):
        fv = None
    return Event(
        title=title,
        start_datetime_iso=str(d.get('start_datetime_iso') or '').strip(),
        end_datetime_iso=str(d.get('end_datetime_iso') or '').strip(),
        location=str(d.get('location') or '').strip(),
        is_free=fv,
        price_notes=str(d.get('price_notes') or '').strip(),
        source_url=str(d.get('source_url') or '').strip(),
        summary=str(d.get('summary') or '').strip(),
        tags=tags,
    )


def dedupe_key(e: Event):
    return e.title.lower(), e.start_datetime_iso, e.location.lower()

norm = []
seen = set()
for d in events:
    ev = normalize_event(d)
    if ev is None:
        continue
    key = dedupe_key(ev)
    if key in seen:
        continue
    seen.add(key)
    norm.append(ev)

print("After filtering + dedupe:", len(norm), "events")


## Personalized markdown plan


In [ ]:
USER_PROFILE = os.getenv(
    'USER_PROFILE',
    (
        'I want technology-focused events: software, AI/ML, data, cloud, product, '
        'startups, and engineering talks. Prefer daytime or early evening; skip nightlife.'
    ),
)

def format_datetime_human(iso_s: str) -> str:
    """RFC3339-ish → e.g. Saturday, 2 May 2026, 09:00 UTC (always shown in UTC)."""
    s = (iso_s or "").strip()
    if not s:
        return "DATE TBD"
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    try:
        dt = datetime.fromisoformat(s)
    except ValueError:
        return s
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    dt_utc = dt.astimezone(timezone.utc)
    return (
        f"{dt_utc:%A}, {dt_utc.day} {dt_utc:%B %Y}, {dt_utc:%H:%M} UTC"
    )


PLANNER_SYSTEM = (
"You are an event concierge. Produce concise markdown with:\n"
"## Executive summary\n"
"## Suggested schedule (rolling 7-day window from today UTC)\n"
"## If I attend only one or two\n"
"## Assumptions + missing data\n\n"
"Requirements:\n"
"- Mention title, tentative date/time, location, free vs paid cues\n"
"- Use human-readable date/time only (weekday, day month year, time, timezone), e.g. "
"“Saturday, 3 May 2026, 14:30 UTC”. Do not print raw ISO-8601 timestamps.\n"
"- Tie rationale back explicitly to USER_PROFILE bullets\n"
"- Label DATE TBD when timing is ambiguous\n"
"- Under **Suggested schedule**, include a day heading or subsection **only** if that calendar "
"day has at least one event listed in EVENTS. Skip empty days entirely—do not leave gaps "
"filler paragraphs.\n"
"- Never write lines like \"No events scheduled\", \"Nothing today\", \"No plans\", "
"or other placeholders for days without events.\n"
)


def events_payload(evts: list[Event]) -> str:
    payloads = []
    for ev in evts:
        end_iso = ev.end_datetime_iso.strip()
        payloads.append(json.dumps({
            "title": ev.title,
            "start": format_datetime_human(ev.start_datetime_iso),
            "end": format_datetime_human(end_iso) if end_iso else "",
            "location": ev.location,
            "free": ev.is_free,
            "price_notes": ev.price_notes,
            "source_url": ev.source_url,
            "summary": ev.summary,
            "tags": ev.tags,
        }, ensure_ascii=False))
    return '\n'.join(payloads)


today_utc = datetime.now(timezone.utc)
today_human = format_datetime_human(today_utc.isoformat())
plan_prompt = (
    f'USER_PROFILE:\n{USER_PROFILE}\n\nTODAY (UTC): {today_human}\n\n'
    'Only list calendar days in the schedule when they match an event below '
    '(skip blank days).\n\n'
    'EVENTS:\n'
    + events_payload(norm)
)

plan_md = llm_chat(PLANNER_SYSTEM, plan_prompt)
display(Markdown(plan_md))
